In [41]:
# LIBRARIES
# STANDARD 
import os
import time
import shutil
from pathlib import Path
from itertools import product
import urllib.parse
import keyring

# --- Third-Party Libraries ---
import pandas as pd
import urllib3

# Tableau
from tableau_api_lib import TableauServerConnection
from tableau_api_lib.utils import querying, flatten_dict_column
import tableauserverclient as TSC

# PowerPoint
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.dml.color import RGBColor
from pptx.oxml.xmlchemy import OxmlElement
from pptx.oxml.ns import qn
import aspose.slides as slides
from aspose.slides import Portion, PortionFormat


In [42]:
# DISGABLE API WARNINGS
# --------------------------------------------------
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [43]:
# PROJECT PATH 
# --------------------------------------------------
code_dir = Path.cwd()
project_root = code_dir.parent

data_dir = project_root / "data"
runs_dir = data_dir / "runs"

In [44]:
# RUN STRUCTURE
# --------------------------------------------------

timestamp = time.strftime("%Y%m%d_%H%M%S")

run_dir = runs_dir / f"run_{timestamp}"
output_dir = run_dir / "outputs"
code_dir = run_dir / "code"

for d in [run_dir, output_dir, code_dir]:
    d.mkdir(parents=True, exist_ok=True)

log_file = run_dir / "logfile.txt"

def log(msg):
    ts = time.strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{ts}] {msg}"
    print(line)
    with open(log_file, "a", encoding="utf-8") as f:
        f.write(line + "\n")

# Save script copy
try:
    shutil.copy(Path.cwd() / "tableau_exporter.ipynb", code_dir / "tableau_export.py")
except Exception:
    pass

log("Tableau export script started")
log(f"Run directory: {run_dir}")


[2026-06-09 14:49:35] Tableau export script started
[2026-06-09 14:49:35] Run directory: c:\Users\medwards\OneDrive - HDR, Inc\sandbox\tableau-rpa\data\runs\run_20260609_144935


In [45]:
# TABLEAU SERVER CLIENT LIBRARY (TSC) - CONFIG & AUTH
# --------------------------------------------------
SERVICE_NAME = "tableau_tsc"

# Retrieve secrets
pat_name = keyring.get_password(SERVICE_NAME, "pat_name")
pat_token = keyring.get_password(SERVICE_NAME, "pat_token")
site_id = keyring.get_password(SERVICE_NAME, "site_id")
server_url = keyring.get_password(SERVICE_NAME, "server_url")

# Auth object
tableau_auth = TSC.PersonalAccessTokenAuth(
    pat_name,
    pat_token,
    site_id
)

server = TSC.Server(server_url, use_server_version=True)
server.add_http_options({'verify': False})
server.auth.sign_in(tableau_auth)
print("Signed in to Tableau Server successfully")


Signed in to Tableau Server successfully


In [46]:
# PUBLISH WORKBOOK

# Input parameter
local_twb_path = r"C:\Users\medwards\OneDrive - HDR, Inc\sandbox\tableau-rpa\data\input\inpatient_ECU.twbx"

# --- Constants ---
project_name = "Tableau_RPA"

# --- Parse workbook name from file ---
file_name = os.path.basename(local_twb_path)
workbook_name = os.path.splitext(file_name)[0]

# --- Find project by paging through results ---
def find_project_by_name(name):
    page_number = 1
    while True:
        req_options = TSC.RequestOptions(pagenumber=page_number)
        projects, pagination = server.projects.get(req_options=req_options)
        for proj in projects:
            if proj.name == name:
                return proj
        if len(projects) < pagination.page_size:
            break
        page_number += 1
    return None

existing_project = find_project_by_name(project_name)

if existing_project:
    project_id = existing_project.id
    print(f"Using existing project: {project_name} ({project_id})")
else:
    new_project_item = TSC.ProjectItem(
        name=project_name,
        content_permissions='LockedToProject',
        description='Temp folder for Tableau RPA exports'
    )
    new_project = server.projects.create(new_project_item)
    project_id = new_project.id
    print(f"Created project: {project_name} ({project_id})")

# --- Delete existing workbooks in the project ---
def delete_workbooks_in_project(project_id):
    page_number = 1
    while True:
        req_options = TSC.RequestOptions(pagenumber=page_number)
        workbooks, pagination = server.workbooks.get(req_options=req_options)
        for wb in workbooks:
            if wb.project_id == project_id:
                print(f"Deleting workbook: {wb.name} ({wb.id})")
                server.workbooks.delete(wb.id)
        if len(workbooks) < pagination.page_size:
            break
        page_number += 1

delete_workbooks_in_project(project_id)

# --- Create workbook item ---
new_workbook = TSC.WorkbookItem(
    name=workbook_name,
    project_id=project_id
)

# --- Publish workbook ---
publish_workbook = server.workbooks.publish(
    new_workbook,
    local_twb_path,
    'CreateNew',
    skip_connection_check=True
)

print(f"Published workbook ID: {publish_workbook.id}")

Using existing project: Tableau_RPA (9f7f9086-7343-4955-b5d6-4fc8330d3e38)
Deleting workbook: inpatient_ECU (2810a3b2-a4b0-4a6a-ad75-1a3152adcc15)
Published workbook ID: 353bd623-da8d-4588-8487-c1592b743675


In [47]:
# TABLEAU API LIBRARY (TAL) - CONFIG & AUTH
# --------------------------------------------------
SERVICE_NAME = "tableau_tsc"

# Retrieve secrets from keyring
pat_name = keyring.get_password(SERVICE_NAME, "pat_name")
pat_token = keyring.get_password(SERVICE_NAME, "pat_token")
site_name = keyring.get_password(SERVICE_NAME, "site_name")
server_url = keyring.get_password(SERVICE_NAME, "server_url")

tableau_config = {
    'tableau_prod': {
        'server': server_url,
        'api_version': '3.27',
        'personal_access_token_name': pat_name,
        'personal_access_token_secret': pat_token,
        'site_name': site_name,
        'site_url': site_name,  
        'cache_buster': '',
        'temp_dir': ''
    }
}

connection = TableauServerConnection(tableau_config, ssl_verify=False)
connection.sign_in()

print("Signed in to Tableau via TAL successfully")

Signed in to Tableau via TAL successfully


In [48]:
# TAL - GET VIEWS DATAFRAME
# --------------------------------------------------
view_df = querying.get_views_dataframe(connection)
views_with_wb_info_df = flatten_dict_column(
    df=view_df,
    keys=["name", "id"],
    col_name="workbook"
)
published_views = views_with_wb_info_df[
    (views_with_wb_info_df["workbook_id"] == publish_workbook.id) 
]

In [49]:
# MAIN LOOP - GENERATE PNGS AND SLIDES
# --------------------------------------------------

# --- CONFIG ---
driver_path = r"C:\Users\medwards\OneDrive - HDR, Inc\sandbox\tableau-rpa\code\tbe_driver.csv"
df_driver = pd.read_csv(driver_path)

pptx_template_path = Path(r"C:\Users\medwards\OneDrive - HDR, Inc\sandbox\tableau-rpa\data\input\20260615_HOSPITAL_TEMPLATE_CurrentState.pptx")
pptx_output_path = output_dir / f"{pptx_template_path.stem}_insert{pptx_template_path.suffix}"

# --- HELPERS ---

def normalize_text(value):
    if pd.isna(value):
        return None
    text = str(value).strip()
    return text if text else None

def parse_quoted_list(value_string):
    if pd.isna(value_string):
        return []
    return [v.strip().strip('"') for v in value_string.split(",")]


def parse_group_ids(group_string):
    if pd.isna(group_string):
        return []
    return [int(x.strip()) for x in str(group_string).split(",")]


def build_groups(values, group_ids):
    if not values or not group_ids:
        return []

    groups = {}
    for val, grp in zip(values, group_ids):
        groups.setdefault(grp, []).append(val)

    return [groups[k] for k in sorted(groups.keys())]


def build_filter(field_name, values):
    field_encoded = urllib.parse.quote(field_name)
    encoded_values = [urllib.parse.quote(v) for v in values]
    joined_vals = ",".join(encoded_values)

    return f"vf_{field_encoded}={joined_vals}"

# --------------------------------------------------
# GENERATE PNGS + TXT
# --------------------------------------------------

image_paths = []
slide_metadata = []

for _, row in df_driver.iterrows():

    story_name = row["story"]
    ppt_name = row["ppt_name"]
    section_name = normalize_text(row.get("ppt_section")) or "Default Section"

    # --- Find Tableau view ---
    view_match = published_views[
        (published_views["sheetType"] == "story") &
        (published_views["name"] == story_name)
    ]

    if view_match.empty:
        log(f"View not found for story: {story_name}")
        continue

    view_id = view_match.iloc[0]["id"]

    # --- Build variable groups ---
    var_groups = []

    for v in ["v1", "v2", "v3"]:
        name = row.get(f"{v}_name")
        values = parse_quoted_list(row.get(f"{v}_values"))
        group_ids = parse_group_ids(row.get(f"{v}_valgroups"))

        if name and values and group_ids:
            groups = build_groups(values, group_ids)
            var_groups.append((name, groups))

    if not var_groups:
        continue

    group_lists = [g for _, g in var_groups]

    # --- Loop through combinations ---
    for combo_idx, combo in enumerate(product(*group_lists), start=1):

        filter_parts = []

        for (field_name, _), group_values in zip(var_groups, combo):
            filter_parts.append(build_filter(field_name, group_values))

        combined_filter = "&".join(filter_parts)

        response = connection.query_view_image(
            view_id=view_id,
            parameter_dict={"filter": combined_filter}
        )

        safe_story = story_name.replace(" ", "_")
        field_part = "_".join([fn.replace(" ", "_") for fn, _ in var_groups])
        gp_num = str(combo_idx).zfill(2)

        file_name = f"{workbook_name}_{safe_story}_{field_part}_gp{gp_num}.png"
        file_path = output_dir / file_name

        with open(file_path, "wb") as f:
            f.write(response.content)

        image_paths.append(file_path)

        # --- Human-readable filter text ---
        readable_filters = [
            f"{field_name}: {', '.join(group_values)}"
            for (field_name, _), group_values in zip(var_groups, combo)
        ]

        param_text_short = "Report Parameters:\n" + "\n".join(readable_filters)

        # ✅ FULL TXT OUTPUT RESTORED
        filter_text = "\n".join([
            f"Workbook: {workbook_name}",
            f"Story: {story_name}",
            f"View ID: {view_id}",
            f"Group Index: {gp_num}",
            "",
            "Filters Applied:",
            *readable_filters,
            "",
            "Raw Filter String:",
            combined_filter,
        ])

        txt_file_name = file_name.replace(".png", ".filter.txt")
        txt_file_path = output_dir / txt_file_name

        with open(txt_file_path, "w", encoding="utf-8") as f:
            f.write(filter_text)

        slide_metadata.append({
            "image_path": file_path,
            "param_text": param_text_short,
            "ppt_section": section_name,
            "ppt_name": ppt_name,
        })

# --------------------------------------------------
# BUILD PPT 
# --------------------------------------------------

if slide_metadata:

    from collections import OrderedDict

    # ✅ STEP 1: Group slides by section (preserves order)
    sectioned = OrderedDict()

    for item in slide_metadata:
        sec = normalize_text(item["ppt_section"]) or "Default Section"
        sectioned.setdefault(sec, []).append(item)

    with slides.Presentation(str(pptx_template_path)) as presentation:

        # --- Layout (match your template layout name) ---
        layout = None
        for l in presentation.layout_slides:
            if l.name == "1_Text":
                layout = l
                break

        if layout is None:
            raise ValueError("Layout '1_Text' not found in template")

        # ✅ STEP 2: Create slides in section order + track first slide per section
        section_first_slide = {}

        for section_name, items in sectioned.items():

            for item in items:

                image_path = str(item["image_path"])
                param_text = item["param_text"]
                ppt_name = item["ppt_name"]

                # ✅ Create slide (ORDER DEFINES SECTION MEMBERSHIP)
                slide = presentation.slides.add_empty_slide(layout)

                # ✅ Capture first slide for section
                if section_name not in section_first_slide:
                    section_first_slide[section_name] = slide

                # --- Title ---
                title_shape = None

                for shape in slide.shapes:
                    if shape.placeholder is not None:
                        if shape.placeholder.type == slides.PlaceholderType.TITLE:
                            title_shape = shape
                            break

                if title_shape is not None:
                    title_shape.text_frame.text = ppt_name

                # --- Param text box ---
                textbox = slide.shapes.add_auto_shape(
                    slides.ShapeType.RECTANGLE,
                    519.12,   # x
                    14.4,     # y
                    425.52,   # width
                    104.4     # height
                )
                #textbox.text_frame.text = param_text
                text_frame = textbox.text_frame
                text_frame.text = ""  
                p = text_frame.paragraphs[0]
                p.paragraph_format.alignment = slides.TextAlignment.LEFT
                portion = Portion()
                portion.text = param_text
                portion.portion_format.font_height = 9 
                p.portions.add(portion)

                # --- Image ---
                with open(image_path, "rb") as img_file:
                    image = presentation.images.add_image(img_file)

                slide_width = presentation.slide_size.size.width

                slide.shapes.add_picture_frame(
                    slides.ShapeType.RECTANGLE,
                    36,        # 0.5 inches
                    118.8,      # 1.2 inches
                    slide_width - 72,
                    400,
                    image
                )

        # ✅ STEP 3: Rebuild sections correctly
        presentation.sections.clear()

        for section_name, first_slide in section_first_slide.items():
            presentation.sections.add_section(section_name, first_slide)

        # --- Save ---
        presentation.save(str(pptx_output_path), slides.export.SaveFormat.PPTX)

    log(f"✅ Saved deck: {pptx_output_path}")

else:
    log("No PNG images were generated, so no slides were inserted.")

[2026-06-09 14:50:37] ✅ Saved deck: c:\Users\medwards\OneDrive - HDR, Inc\sandbox\tableau-rpa\data\runs\run_20260609_144935\outputs\20260615_HOSPITAL_TEMPLATE_CurrentState_insert.pptx


In [50]:
from pptx import Presentation

ppt_path = str(pptx_output_path)

print(f"🔧 Cleaning PPTX with python-pptx: {ppt_path}")

prs = Presentation(ppt_path)

for slide in prs.slides:
    shapes_to_remove = []

    for shape in slide.shapes:
        if shape.has_text_frame:
            text = shape.text_frame.text or ""

            # ✅ Identify watermark text
            if any(w in text.lower() for w in ["evaluation", "aspose", "copyright"]):
                shapes_to_remove.append(shape)

    # ✅ Remove shapes safely
    for shape in shapes_to_remove:
        sp = shape._element
        sp.getparent().remove(sp)

# ✅ Save cleaned file
prs.save(ppt_path)

print("✅ Watermark removed via python-pptx")

🔧 Cleaning PPTX with python-pptx: c:\Users\medwards\OneDrive - HDR, Inc\sandbox\tableau-rpa\data\runs\run_20260609_144935\outputs\20260615_HOSPITAL_TEMPLATE_CurrentState_insert.pptx
✅ Watermark removed via python-pptx
